# Episode 1 — PRNG Keys, Weight + Bias Dict, and Broadcasting

**Instructor notebook** · run top-to-bottom before recording.

Explicit randomness with splittable keys, a nested `params` dict for one affine layer, and broadcasting for the bias.

| | |
|---|---|
| **Prereq** | Episode 0 |
| **Next** | Episode 2 — `jit`, `vmap`, jaxpr, and timing matmul |

**JAX docs:** [Pseudorandom numbers](https://docs.jax.dev/en/latest/random-numbers.html) · [NumPy broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html) · [Pytrees](https://docs.jax.dev/en/latest/pytrees.html)

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr

## PRNG keys

JAX has no hidden global RNG. Create a **key**, **split** before each random draw, and never reuse a consumed subkey.

In [2]:
key = jr.key(0)
key, subkey = jr.split(key)

a = jr.normal(subkey, (3,))
print(a)

# Same root key → same draws
key = jr.key(0)
key, subkey = jr.split(key)
b = jr.normal(subkey, (3,))
print(jnp.allclose(a, b))

[-2.4424558  -2.0356805   0.20554423]
True


## Nested `params` and `init_params`

Store weights in a **nested dict** (a small pytree). Shapes: `x` is `(B, D_in)`, `W` is `(D_in, D_out)`, `b` is `(D_out,)`.

In [3]:
def init_params(key, d_in, d_out):
    key, k_w, k_b = jr.split(key, 3)
    W = jr.normal(k_w, (d_in, d_out)) * 0.1
    b = jnp.zeros(d_out)
    return {"affine": {"W": W, "b": b}}


d_in, d_out = 10, 10
params = init_params(jr.key(42), d_in, d_out)
print(params["affine"]["W"].shape, params["affine"]["b"].shape)

(10, 10) (10,)


## Pytrees (preview)

A **pytree** is any nested structure JAX can walk — typically `dict`, `list`, or `tuple` nodes whose **leaves** are the actual data (arrays, scalars, or other JAX types).

Our `params` dict is a small pytree: the outer `"affine"` key holds another dict; the leaves are `W` and `b`. JAX utilities recurse through the nesting and only touch the leaves.

**JAX docs:** [Pytrees](https://docs.jax.dev/en/latest/pytrees.html)

In [4]:
leaves = jax.tree.leaves(params)
print("number of leaves:", len(leaves))
print("leaf shapes:", [leaf.shape for leaf in leaves])

number of leaves: 2
leaf shapes: [(10, 10), (10,)]


## `jax.tree.map`

`jax.tree.map` applies a function to **every leaf** in a pytree, preserving the nested structure. Here we add `0.01` to each array in `params`.

In [5]:
print("before W[0,0]:", params["affine"]["W"][0, 0])
params = jax.tree.map(lambda a: a + 0.01, params)
print("after  W[0,0]:", params["affine"]["W"][0, 0])
print("after  b[0]:", params["affine"]["b"][0])

before W[0,0]: 0.060576405
after  W[0,0]: 0.07057641
after  b[0]: 0.01


## Forward — `y = x @ W + b`

In [6]:
def forward(params, x):
    W = params["affine"]["W"]
    b = params["affine"]["b"]
    return x @ W + b


B = 64
key = jr.key(1)
key, k_x = jr.split(key)
x = jr.normal(k_x, (B, d_in))

y = forward(params, x)
print("y.shape:", y.shape)

y.shape: (64, 10)


## Broadcasting

JAX follows [NumPy broadcasting rules](https://numpy.org/doc/stable/user/basics.broadcasting.html): compare shapes from the **right**; dimensions match if equal or one is `1`.

`x @ W` has shape `(B, D_out)`. `b` has shape `(D_out,)` — it broadcasts across batch rows.

In [7]:
print(jnp.broadcast_shapes((B, d_out), (d_out,)))

logits = x @ params["affine"]["W"]
print(logits.shape)
y_broadcast = logits + params["affine"]["b"]
print(jnp.allclose(y_broadcast, y))

(64, 10)
(64, 10)
True


---
## Exercise

Implement a **2-layer MLP**:

1. `init_mlp_params(key, d_in, d_hidden, d_out)` — one function that `split`s subkeys and returns a pytree `{0: {"W", "b"}, 1: {"W", "b"}}` with **int** layer indices.
2. Loop over layer indices and apply `x @ W + b` at each step (no separate forward function).
3. Run on batch input `x` with shape `(B, D_in)` and print the **final activation** shape `(B, D_out)`.

*(Solution below.)*

In [9]:
def init_mlp_params(key, d_in, d_hidden, d_out):
    key, k0, k1 = jr.split(key, 3)
    k0, k0_w = jr.split(k0, 2)
    k1, k1_w = jr.split(k1, 2)
    return {
        0: {
            "W": jr.normal(k0_w, (d_in, d_hidden)) * 0.1,
            "b": jnp.zeros(d_hidden),
        },
        1: {
            "W": jr.normal(k1_w, (d_hidden, d_out)) * 0.1,
            "b": jnp.zeros(d_out),
        },
    }


d_hidden = 16
mlp_params = init_mlp_params(jr.key(7), d_in, d_hidden, d_out)

B_ex = 12
key, k_x = jr.split(jr.key(8))
x_ex = jr.normal(k_x, (B_ex, d_in))

activation = x_ex
for layer_idx in range(len(mlp_params)):
    W = mlp_params[layer_idx]["W"]
    b = mlp_params[layer_idx]["b"]
    activation = activation @ W + b

print("final activation shape:", activation.shape)

final activation shape: (12, 10)


**Next:** Episode 2 — `jit`, `vmap`, inspect jaxprs, and `timeit` matmul.